# Undiagnosed Hypertension and Diabetes Screening with NHANES
## Part 2: Features, baselines, models and test-set performance

Upload `nhanes_analytic.csv` (saved by Part 1) when the first cell asks for it.

**What this part does**
1. Builds laboratory-free predictor sets a primary care clinic could collect
2. Trains logistic regression, XGBoost, LightGBM and an Explainable Boosting Machine
3. Chooses model settings on the 2017-2020 validation cycle, refits on 2011-2020, then tests once on 2021-2023
4. Reports AUC (95% bootstrap CI), AUPRC, Brier score, calibration intercept and slope, and screening sensitivity/specificity
5. Compares the diabetes models with the ADA risk score on 2017-2020, where family history is still available

Race/ethnicity and income are deliberately not used as predictors (they do not transfer to a Ghanaian clinic);
they are kept for the fairness analysis in Part 3.

### Step 2.1: Setup and data

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost", "lightgbm", "interpret"], check=False)

import os, re, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from interpret.glassbox import ExplainableBoostingClassifier

warnings.filterwarnings("ignore")
SEED = 2026
np.random.seed(SEED)
OUT = "/content/part2_out" if os.path.isdir("/content") else "./part2_out"
os.makedirs(OUT, exist_ok=True)

CSV = "/content/nhanes_analytic.csv" if os.path.isdir("/content") else "nhanes_analytic.csv"
if not os.path.exists(CSV):
    from google.colab import files
    print("Upload nhanes_analytic.csv from Part 1")
    up = files.upload()
    CSV = "/content/" + list(up.keys())[0]
data = pd.read_csv(CSV)
print(data.shape)

### Step 2.2: Predictor sets
- **Diabetes, main model:** usable in every cycle, so it can be tested on 2021-2023
- **Diabetes, extended model:** adds family history and gestational diabetes (not asked in 2021-2023), evaluated on 2017-2020 only
- **Hypertension:** blood pressure is excluded because it defines the outcome

In [ ]:
data["hx_htn"] = data["told_htn"]                      # ever told high blood pressure (1/0)
data["hx_dm"] = (data["told_dm"] == 1).astype(float)   # ever told diabetes
data.loc[data["told_dm"].isna(), "hx_dm"] = np.nan
data["smoking"] = data["smoking"].fillna("missing")

NUM_DM = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sbp_mean", "dbp_mean", "sedentary_min"]
BIN_DM = ["sex_female", "hx_htn", "cvd_history", "active_leisure"]
CAT_DM = ["smoking", "education"]
NUM_HTN = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sedentary_min"]
BIN_HTN = ["sex_female", "hx_dm", "cvd_history", "active_leisure"]
CAT_HTN = ["smoking", "education"]
EXTRA_DM = ["famhx_dm", "gdm"]

TASKS = {
    "diabetes": {"pop": "dm_at_risk_pop", "y": "undx_dm", "num": NUM_DM, "bin": BIN_DM, "cat": CAT_DM,
                 "simple": ["age", "sex_female", "bmi"]},
    "hypertension": {"pop": "htn_at_risk_pop", "y": "undx_htn", "num": NUM_HTN, "bin": BIN_HTN, "cat": CAT_HTN,
                     "simple": ["age", "sex_female", "bmi"]},
}

def task_data(task):
    t = TASKS[task]
    d = data[(data[t["pop"]] == 1) & data[t["y"]].notna()].copy()
    d["education"] = d["education"].map(lambda v: "missing" if pd.isna(v) else str(int(v)))
    return d

for task in TASKS:
    d = task_data(task)
    print(task, d.groupby("split")[TASKS[task]["y"]].agg(["size", "sum", "mean"]).round(4).to_dict())

### Step 2.3: Models
Tree models handle missing values natively; logistic regression and EBM use median/mode imputation
fitted on the training data only. No resampling is used, because oversampling distorts calibration.

In [ ]:
def preprocessor(num, bin_, cat, scale=True):
    num_pipe = [("imp", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_pipe.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_pipe), num),
        ("bin", SimpleImputer(strategy="most_frequent"), bin_),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", drop="first"))]), cat),
    ])

def tree_matrix(d, num, bin_, cat):
    X = d[num + bin_].copy()
    for c in cat:
        X = pd.concat([X, pd.get_dummies(d[c], prefix=c, dtype=float)], axis=1)
    # XGBoost rejects [, ] and < in column names, so keep names to letters, digits and underscores
    X.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", str(c)) for c in X.columns]
    return X

def make_model(name, params, num, bin_, cat):
    if name == "LR":
        return Pipeline([("pre", preprocessor(num, bin_, cat)), ("m", LogisticRegression(max_iter=2000, C=params.get("C", 1.0)))])
    if name == "EBM":
        return Pipeline([("pre", preprocessor(num, bin_, cat, scale=False)),
                         ("m", ExplainableBoostingClassifier(random_state=SEED, interactions=params.get("interactions", 10)))])
    if name == "XGBoost":
        return XGBClassifier(n_estimators=params["n"], max_depth=params["depth"], learning_rate=params["lr"],
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=5, eval_metric="logloss",
                             random_state=SEED, n_jobs=-1)
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=params["n"], num_leaves=params["leaves"], learning_rate=params["lr"],
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, min_child_samples=30,
                              random_state=SEED, verbose=-1)

GRIDS = {
    "LR": [{"C": c} for c in [0.01, 0.1, 1.0]],
    "EBM": [{"interactions": i} for i in [0, 10]],
    "XGBoost": [{"n": n, "depth": dp, "lr": lr} for n in [200, 500] for dp in [2, 3, 4] for lr in [0.03]],
    "LightGBM": [{"n": n, "leaves": lv, "lr": lr} for n in [200, 500] for lv in [7, 15] for lr in [0.03]],
}
TREE = {"XGBoost", "LightGBM"}

def fit_predict(name, params, tr, te, num, bin_, cat, y):
    m = make_model(name, params, num, bin_, cat)
    if name in TREE:
        Xtr = tree_matrix(tr, num, bin_, cat)
        Xte = tree_matrix(te, num, bin_, cat).reindex(columns=Xtr.columns, fill_value=0)
        m.fit(Xtr, tr[y])
        return m, m.predict_proba(Xte)[:, 1]
    cols = num + bin_ + cat
    m.fit(tr[cols], tr[y])
    return m, m.predict_proba(te[cols])[:, 1]

### Step 2.4: Metrics

In [ ]:
def calib_int_slope(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp, y).coef_[0][0]
    # calibration-in-the-large: intercept with the linear predictor as offset
    import statsmodels.api as sm
    inter = sm.GLM(y, np.ones_like(lp), family=sm.families.Binomial(), offset=lp.ravel()).fit().params[0]
    return inter, slope

def boot_auc(y, p, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    aucs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        aucs.append(roc_auc_score(y[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

def threshold_for_sensitivity(y, p, target=0.80):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr >= target)]

def evaluate(y, p, thr):
    y = np.asarray(y)
    lo, hi = boot_auc(y, p)
    inter, slope = calib_int_slope(y, p)
    flag = p >= thr
    tp, fp = (flag & (y == 1)).sum(), (flag & (y == 0)).sum()
    fn, tn = (~flag & (y == 1)).sum(), (~flag & (y == 0)).sum()
    return {"AUC": roc_auc_score(y, p), "AUC_lo": lo, "AUC_hi": hi, "AUPRC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p), "Cal_intercept": inter, "Cal_slope": slope,
            "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp), "PPV": tp / max(tp + fp, 1),
            "Flagged_%": 100 * flag.mean()}

### Step 2.5: Tune on 2017-2020, refit on 2011-2020, test once on 2021-2023

In [ ]:
results, preds, best_params = [], {}, {}
for task, t in TASKS.items():
    d = task_data(task)
    tr, va, te = d[d.split == "train"], d[d.split == "validation"], d[d.split == "test"]
    trva = pd.concat([tr, va])
    y = t["y"]
    # Simple baseline: logistic regression on age, sex and BMI
    simple_num, simple_bin = ["age", "bmi"], ["sex_female"]
    _, pv = fit_predict("LR", {"C": 1.0}, tr, va, simple_num, simple_bin, [], y)
    thr = threshold_for_sensitivity(va[y], pv)
    _, pt = fit_predict("LR", {"C": 1.0}, trva, te, simple_num, simple_bin, [], y)
    preds[(task, "Baseline LR (age, sex, BMI)")] = (te[y].values, pt)
    results.append({"task": task, "model": "Baseline LR (age, sex, BMI)", **evaluate(te[y], pt, thr)})
    for name, grid in GRIDS.items():
        scores = []
        for params in grid:
            _, pv = fit_predict(name, params, tr, va, t["num"], t["bin"], t["cat"], y)
            scores.append((roc_auc_score(va[y], pv), params, pv))
        auc_v, params, pv = max(scores, key=lambda s: s[0])
        best_params[(task, name)] = params
        thr = threshold_for_sensitivity(va[y], pv)
        model, pt = fit_predict(name, params, trva, te, t["num"], t["bin"], t["cat"], y)
        preds[(task, name)] = (te[y].values, pt)
        results.append({"task": task, "model": name, "validation_AUC": auc_v, **evaluate(te[y], pt, thr)})
        print(f"{task:12s} {name:9s} best {params}  validation AUC {auc_v:.3f}  test AUC {results[-1]['AUC']:.3f}")

res = pd.DataFrame(results)
cols = ["task", "model", "validation_AUC", "AUC", "AUC_lo", "AUC_hi", "AUPRC", "Brier", "Cal_intercept", "Cal_slope",
        "Sensitivity", "Specificity", "PPV", "Flagged_%"]
res = res[cols].round(3)
res.to_csv(os.path.join(OUT, "test_performance.csv"), index=False)
print(res.to_string(index=False))

### Step 2.6: Diabetes extended model versus the ADA risk score (2017-2020)
The ADA risk test (Bang et al., 2009) scores age, sex, gestational diabetes, family history,
history of hypertension, physical activity and BMI. It needs family history, so it is compared on 2017-2020.

In [ ]:
def ada_score(d):
    age_pts = pd.cut(d["age"], [0, 39.99, 49.99, 59.99, 200], labels=[0, 1, 2, 3]).astype(float)
    bmi_pts = pd.cut(d["bmi"], [0, 24.99, 29.99, 39.99, 200], labels=[0, 1, 2, 3]).astype(float)
    s = (age_pts + (1 - d["sex_female"]) + d["gdm"].fillna(0) + d["famhx_dm"].fillna(0)
         + d["hx_htn"].fillna(0) + (1 - d["active_leisure"].fillna(1)) + bmi_pts.fillna(0))
    return s

d = task_data("diabetes")
tr, va = d[d.split == "train"], d[d.split == "validation"]
ext_rows = []
ada = ada_score(va).values
ext_rows.append({"model": "ADA risk score (points)", **evaluate(va["undx_dm"], ada / 10, 5 / 10)})
for name in ["LR", "XGBoost"]:
    params = best_params[("diabetes", name)]
    for label, extra in [("main", []), ("extended (+ family history, GDM)", EXTRA_DM)]:
        _, pv = fit_predict(name, params, tr, va, NUM_DM, BIN_DM + extra, CAT_DM, "undx_dm")
        thr = threshold_for_sensitivity(va["undx_dm"], pv)
        ext_rows.append({"model": f"{name} {label}", **evaluate(va["undx_dm"], pv, thr)})
ext = pd.DataFrame(ext_rows).round(3)
ext.to_csv(os.path.join(OUT, "ada_comparison_2017_2020.csv"), index=False)
print("Note: calibration values for the ADA score are not meaningful (it is a points score, not a probability).")
print(ext.to_string(index=False))

### Step 2.7: Figures (ROC and calibration on the 2021-2023 test cycle)

In [ ]:
from sklearn.calibration import calibration_curve
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for r, task in enumerate(TASKS):
    for (tk, name), (yt, pt) in preds.items():
        if tk != task:
            continue
        fpr, tpr, _ = roc_curve(yt, pt)
        axes[r, 0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(yt, pt):.3f})")
        frac, mean_p = calibration_curve(yt, pt, n_bins=10, strategy="quantile")
        axes[r, 1].plot(mean_p, frac, marker="o", label=name)
    axes[r, 0].plot([0, 1], [0, 1], "k--", lw=0.8)
    axes[r, 0].set(title=f"ROC, undiagnosed {task} (2021-2023)", xlabel="1 - specificity", ylabel="Sensitivity")
    axes[r, 0].legend(fontsize=8)
    lim = max(axes[r, 1].get_xlim()[1], axes[r, 1].get_ylim()[1])
    axes[r, 1].plot([0, lim], [0, lim], "k--", lw=0.8)
    axes[r, 1].set(title=f"Calibration, undiagnosed {task} (2021-2023)", xlabel="Predicted risk", ylabel="Observed proportion")
    axes[r, 1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "fig_roc_calibration.png"), dpi=300)
plt.show()

### Step 2.8: Save predictions for Part 3 (decision curves, fairness and SHAP stability)

In [ ]:
rows = []
for task in TASKS:
    d = task_data(task)
    te = d[d.split == "test"]
    out = te[["SEQN", "cycle", "split", "wt_pooled", "age", "sex_female", "race_eth", "pir", TASKS[task]["y"]]].copy()
    out = out.rename(columns={TASKS[task]["y"]: "y"})
    for (tk, name), (yt, pt) in preds.items():
        if tk == task:
            out[name] = pt
    out["task"] = task
    rows.append(out)
pd.concat(rows).to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)
json.dump({f"{k[0]}|{k[1]}": v for k, v in best_params.items()}, open(os.path.join(OUT, "best_params.json"), "w"), indent=1)
print("Saved to", OUT)
try:
    from google.colab import files
    for f in ["test_performance.csv", "ada_comparison_2017_2020.csv", "fig_roc_calibration.png", "test_predictions.csv", "best_params.json"]:
        files.download(os.path.join(OUT, f))
except Exception:
    pass

### What to send back
The printed tables from Steps 2.5 and 2.6 and the figure from Step 2.7.